# CpGPT Embedding Extraction
**Uses the CpGPT-2M-Cancer finetuned model from HuggingFace**

Extracts 128-dimensional sample embeddings from TCGA methylation data using CpGPT,
a foundation model for DNA methylation. The cancer model also outputs cancer probability.


In [1]:
# Step 1 — Install CpGPT and dependencies
!pip install -q CpGPT anndata huggingface_hub
!pip install -q pyarrow==20.0.0

import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')

GPU available: True
GPU name     : Tesla T4


In [1]:
# Step 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
# Step 3 — Configuration
# UPDATE THIS to your shared Google Drive folder name
SHARED_FOLDER_NAME = "multiomics-project"

from pathlib import Path

BASE          = Path(f"/content/drive/MyDrive/{SHARED_FOLDER_NAME}")
INPUT_H5AD    = BASE / "data/processed/tcga_methylation.h5ad"
OUTPUT_EMB    = BASE / "data/processed/cpgpt_embeddings.npy"
LABELS_PATH   = BASE / "data/processed/geneformer_labels.npy"

# CpGPT-specific paths (will be populated by download step)
DEPS_DIR      = Path("/content/dependencies")
MODEL_DIR     = DEPS_DIR / "model"
HUMAN_DIR     = DEPS_DIR / "human"
DATA_DIR      = Path("/content/data")
PROCESSED_DIR = DATA_DIR / "tcga/processed"

MODEL_NAME            = "cancer"
MODEL_CHECKPOINT_PATH = MODEL_DIR / f"weights/{MODEL_NAME}.ckpt"
MODEL_CONFIG_PATH     = MODEL_DIR / f"config/{MODEL_NAME}.yaml"
MODEL_VOCAB_PATH      = MODEL_DIR / f"vocab/{MODEL_NAME}.json"

MAX_INPUT_LENGTH = 20_000  # max CpG sites per sample (CpGPT handles missing data natively)
BATCH_SIZE       = 4       # cancer config default

print('Methylation h5ad exists :', INPUT_H5AD.exists())
print('Labels file exists      :', LABELS_PATH.exists())

Methylation h5ad exists : True
Labels file exists      : True


In [2]:
# Step 4 — Download CpGPT model and human dependencies from HuggingFace
# This downloads ~6 GB total (DNA embeddings + metadata + model weights)
# Takes ~5-10 min on Colab. Files are cached — skips if already downloaded.

from huggingface_hub import snapshot_download
import os

# Download model weights, config, and vocabulary
if not MODEL_CHECKPOINT_PATH.exists():
    print('Downloading CpGPT model files from HuggingFace...')
    snapshot_download(
        repo_id="lucascamillomd/cpgpt-models",
        local_dir=str(MODEL_DIR),
    )
    print('Model files downloaded!')
else:
    print('Model files already exist, skipping download.')

# Download human dependencies (DNA embeddings, Ensembl metadata, Illumina metadata)
if not HUMAN_DIR.exists() or not (HUMAN_DIR / "illumina_metadata.db").exists():
    print('Downloading CpGPT human dependencies from HuggingFace (~6 GB)...')
    snapshot_download(
        repo_id="lucascamillomd/cpgpt-human-dependencies",
        local_dir=str(HUMAN_DIR),
    )
    print('Human dependencies downloaded!')
else:
    print('Human dependencies already exist, skipping download.')

print()
print('Model checkpoint :', MODEL_CHECKPOINT_PATH.exists())
print('Model config     :', MODEL_CONFIG_PATH.exists())
print('Model vocab      :', MODEL_VOCAB_PATH.exists())
print('Human deps dir   :', HUMAN_DIR.exists())

Model files already exist, skipping download.
Human dependencies already exist, skipping download.

Model checkpoint : True
Model config     : True
Model vocab      : True
Human deps dir   : True


In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import anndata as ad
import torch
import psutil, os, gc

def mem(label):
    proc = psutil.Process(os.getpid())
    rss = proc.memory_info().rss / 1e9
    avail = psutil.virtual_memory().available / 1e9
    print(f"[{label}] RSS: {rss:.2f} GB | Available: {avail:.2f} GB", flush=True)

mem("start")

# -----------------------------
# Load methylation data
# -----------------------------
print("Loading methylation data...")
adata = ad.read_h5ad(INPUT_H5AD)
mem("after read_h5ad")

X = adata.X
if not isinstance(X, np.ndarray):
    X = X.toarray()
X = X.astype(np.float32)  # cast down from float64 here
mem("after toarray+astype")

df = pd.DataFrame(X, columns=adata.var_names, index=adata.obs_names)
df.index.name = "sample_id"
mem("after DataFrame build")

del adata, X
gc.collect()
mem("after cleanup of adata/X")

ARROW_DIR = DATA_DIR / "tcga/raw"
ARROW_DIR.mkdir(parents=True, exist_ok=True)
ARROW_PATH = ARROW_DIR / "tcga_methylation.arrow"
df.to_feather(ARROW_PATH)
mem("after to_feather")

del df
gc.collect()
mem("after cleanup of df")

# -----------------------------
# CpGPT pipeline
# -----------------------------
from cpgpt.data.components.dna_llm_embedder import DNALLMEmbedder
from cpgpt.data.components.illumina_methylation_prober import IlluminaMethylationProber
from cpgpt.data.components.cpgpt_datasaver import CpGPTDataSaver

print("Initializing DNALLMEmbedder...")
embedder = DNALLMEmbedder(dependencies_dir=str(HUMAN_DIR))
mem("after DNALLMEmbedder init")

print("Initializing IlluminaMethylationProber...")
prober = IlluminaMethylationProber(dependencies_dir=str(HUMAN_DIR), embedder=embedder)
mem("after IlluminaMethylationProber init")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Initializing CpGPTDataSaver...")
datasaver = CpGPTDataSaver(data_paths=str(ARROW_PATH), processed_dir=str(PROCESSED_DIR))
mem("after CpGPTDataSaver init")

print("Calling process_files (this is the actual processing step)...")
datasaver.process_files(prober=prober, embedder=embedder)
mem("after process_files")

print("Done.")

[start] RSS: 0.64 GB | Available: 11.66 GB
Loading methylation data...
[after read_h5ad] RSS: 0.98 GB | Available: 11.31 GB
[after toarray+astype] RSS: 1.13 GB | Available: 11.15 GB
[after DataFrame build] RSS: 1.13 GB | Available: 11.15 GB
[after cleanup of adata/X] RSS: 0.82 GB | Available: 11.44 GB
[after to_feather] RSS: 1.39 GB | Available: 10.95 GB
[after cleanup of df] RSS: 1.14 GB | Available: 11.17 GB
Initializing DNALLMEmbedder...
cpgpt -DNALLMEmbedder: Initializing class DNALLMEmbedder.
cpgpt -DNALLMEmbedder: Genome files will be stored under /content/dependencies/human/genomes.
cpgpt -DNALLMEmbedder: DNA embeddings will be stored under /content/dependencies/human/dna_embeddings and subdirectories.
cpgpt -DNALLMEmbedder: Ensembl metadata dictionary loaded successfully
[after DNALLMEmbedder init] RSS: 7.50 GB | Available: 4.87 GB
Initializing IlluminaMethylationProber...
cpgpt -IlluminaMethylationProber: Initializing class IlluminaMethylationProber.
cpgpt -IlluminaMethylation

In [4]:
import gc
# datasaver and the Arrow file are done with — keep prober/embedder if Step 6 needs them, drop datasaver
del datasaver
gc.collect()
mem("after dropping datasaver")

[after dropping datasaver] RSS: 7.64 GB | Available: 4.69 GB


In [5]:
# Step 6 — Load CpGPT Cancer model
# The cancer model is finetuned from CpGPT-2M for cancer vs normal classification.
# Architecture: d_embedding=128, 8 heads, 8 layers, SwiGLU, rotary PE, CLS embedding.
# Must use 16-mixed precision (model was trained with it).

mem("before Step 6 imports")

from cpgpt.infer.cpgpt_inferencer import CpGPTInferencer
from lightning.pytorch import seed_everything

seed_everything(42, workers=True)
mem("after seed_everything")

inferencer = CpGPTInferencer(dependencies_dir=str(DEPS_DIR), data_dir=str(DATA_DIR))
mem("after CpGPTInferencer init")

config = inferencer.load_cpgpt_config(str(MODEL_CONFIG_PATH))
mem("after load_cpgpt_config")
print('Config loaded.')
print(f'  d_embedding: {config.model.net.d_embedding}')
print(f'  n_layers:    {config.model.net.n_layers}')
print(f'  n_heads:     {config.model.net.n_attention_heads}')
print(f'  dna_llm:     {config.data.dna_llm}')

model = inferencer.load_cpgpt_model(config, model_ckpt_path=str(MODEL_CHECKPOINT_PATH), strict_load=True)
mem("after load_cpgpt_model")

print('CpGPT Cancer model loaded and ready!')

[before Step 6 imports] RSS: 7.64 GB | Available: 4.69 GB


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


[after seed_everything] RSS: 7.71 GB | Available: 4.43 GB
cpgpt -CpGPTInferencer: Initializing class CpGPTInferencer.
cpgpt -CpGPTInferencer: Using device: cuda.
cpgpt -CpGPTInferencer: Using dependencies directory: /content/dependencies
cpgpt -CpGPTInferencer: Using data directory: /content/data
cpgpt -CpGPTInferencer: Failed to get available models: Unable to locate credentials
cpgpt -CpGPTInferencer: Failed to get available datasets: Unable to locate credentials
[after CpGPTInferencer init] RSS: 7.72 GB | Available: 4.40 GB
cpgpt -CpGPTInferencer: Loaded CpGPT model config.
[after load_cpgpt_config] RSS: 7.72 GB | Available: 4.40 GB
Config loaded.
  d_embedding: 128
  n_layers:    8
  n_heads:     8
  dna_llm:     nucleotide-transformer-v2-500m-multi-species
cpgpt -CpGPTInferencer: Instantiated CpGPT model from config.
cpgpt -CpGPTInferencer: Using device: cuda.
cpgpt -CpGPTInferencer: Loading checkpoint from: /content/dependencies/model/weights/cancer.ckpt
cpgpt -CpGPTInferencer: C

In [6]:
import inspect
from cpgpt.data.cpgpt_datamodule import CpGPTDataModule
print(inspect.signature(CpGPTDataModule.__init__))

(self, train_dir: str | None = None, val_dir: str | None = None, test_dir: str | None = None, predict_dir: str | None = None, dependencies_dir: str = 'dependencies', batch_size: int = 4, num_workers: int = 4, max_length: int = 10000, dna_llm: str = 'nucleotide-transformer-v2-500m-multi-species', dna_context_len: int = 2001, sorting_strategy: str = 'random', pin_memory: bool = False) -> None


In [7]:
# Step 7 — Extract embeddings via CpGPTTrainer
import gc
import numpy as np

def mem(label):
    proc = psutil.Process(os.getpid())
    rss = proc.memory_info().rss / 1e9
    avail = psutil.virtual_memory().available / 1e9
    print(f"[{label}] RSS: {rss:.2f} GB | Available: {avail:.2f} GB", flush=True)

mem("start of Step 7")

# Free Step 5's embedder/prober — CpGPTDataModule builds its own internally,
# so these are now dead weight stacking on top of the new ones
for name in ["embedder", "prober", "datasaver"]:
    if name in globals():
        del globals()[name]
gc.collect()
mem("after freeing Step 5 embedder/prober/datasaver")

from cpgpt.data.cpgpt_datamodule import CpGPTDataModule
from cpgpt.trainer.cpgpt_trainer import CpGPTTrainer

mem("before CpGPTDataModule init")

# Create data module for prediction
datamodule = CpGPTDataModule(
    predict_dir=str(PROCESSED_DIR),
    dependencies_dir=str(HUMAN_DIR),
    batch_size=BATCH_SIZE,
    num_workers=2,
    max_length=MAX_INPUT_LENGTH,
    dna_llm=config.data.dna_llm,
    dna_context_len=config.data.dna_context_len,
    sorting_strategy=config.data.sorting_strategy,
    pin_memory=False,
)
mem("after CpGPTDataModule init")

# Create trainer with 16-mixed precision (required for correct results)
trainer = CpGPTTrainer(precision="16-mixed")
mem("after CpGPTTrainer init")

print(f'Extracting embeddings for 800 samples (batch={BATCH_SIZE}, max_length={MAX_INPUT_LENGTH})...')

# Run inference — get both sample embeddings and cancer predictions
results = trainer.predict(
    model=model,
    datamodule=datamodule,
    predict_mode="forward",
    return_keys=["sample_embedding", "condition_output"],
)
mem("after trainer.predict")

# Concatenate results
sample_embeddings = []
cancer_outputs = []

for batch_result in results:
    if isinstance(batch_result, dict):
        if "sample_embedding" in batch_result:
            emb = batch_result["sample_embedding"]
            if hasattr(emb, 'cpu'):
                emb = emb.cpu().numpy()
            sample_embeddings.append(emb)
        if "condition_output" in batch_result:
            out = batch_result["condition_output"]
            if hasattr(out, 'cpu'):
                out = out.cpu().numpy()
            cancer_outputs.append(out)
    elif isinstance(batch_result, (list, tuple)):
        for item in batch_result:
            if hasattr(item, 'cpu'):
                item = item.cpu().numpy()
            sample_embeddings.append(item)

embeddings = np.concatenate(sample_embeddings, axis=0)
mem("after concatenating results")
print(f'Embeddings extracted! Shape: {embeddings.shape}')

if cancer_outputs:
    cancer_logits = np.concatenate(cancer_outputs, axis=0)
    cancer_probs = 1 / (1 + np.exp(-cancer_logits.flatten()))  # sigmoid
    print(f'Cancer predictions: {cancer_probs.shape}')
    print(f'  Mean cancer probability: {cancer_probs.mean():.4f}')
    print(f'  Range: [{cancer_probs.min():.4f}, {cancer_probs.max():.4f}]')

[start of Step 7] RSS: 7.82 GB | Available: 4.32 GB
[after freeing Step 5 embedder/prober/datasaver] RSS: 2.99 GB | Available: 9.09 GB
[before CpGPTDataModule init] RSS: 2.99 GB | Available: 9.09 GB
cpgpt -DNALLMEmbedder: Initializing class DNALLMEmbedder.
cpgpt -DNALLMEmbedder: Genome files will be stored under /content/dependencies/human/genomes.
cpgpt -DNALLMEmbedder: DNA embeddings will be stored under /content/dependencies/human/dna_embeddings and subdirectories.
cpgpt -DNALLMEmbedder: Ensembl metadata dictionary loaded successfully
[after CpGPTDataModule init] RSS: 7.68 GB | Available: 4.34 GB


INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[after CpGPTTrainer init] RSS: 7.68 GB | Available: 4.34 GB


INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Extracting embeddings for 800 samples (batch=4, max_length=20000)...
cpgpt -CpGPTDataset: Initializing class CpGPTDataset.
cpgpt -CpGPTDataset: Loaded existing dataset metrics.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

[after trainer.predict] RSS: 8.01 GB | Available: 3.81 GB


In [10]:
import numpy as np

embeddings = results["sample_embedding"]
if hasattr(embeddings, "cpu"):
    embeddings = embeddings.cpu().numpy()

print(f"Embeddings shape: {embeddings.shape}")

# SAVE IMMEDIATELY
EMBEDDINGS_PATH = DATA_DIR / "tcga" / "embeddings_800x128.npy"
np.save(EMBEDDINGS_PATH, embeddings)
print(f"Saved to {EMBEDDINGS_PATH}")

# verify it actually wrote
import os
print(f"File size: {os.path.getsize(EMBEDDINGS_PATH) / 1e6:.2f} MB")

Embeddings shape: (800, 128)
Saved to /content/data/tcga/embeddings_800x128.npy
File size: 0.41 MB


In [13]:
# Step 8 — Save embeddings and validate
import anndata as ad

# Save to Google Drive
np.save(OUTPUT_EMB, embeddings)
print(f'Saved to : {OUTPUT_EMB}')
print(f'Shape    : {embeddings.shape}')
print(f'Any NaN  : {np.isnan(embeddings).any()}')
print(f'Any Inf  : {np.isinf(embeddings).any()}')

# Print summary per cancer type (matching MethylGPT notebook format)
print()
print('=== Final Result ===')
print(f'Total patients : {embeddings.shape[0]}')
print(f'Embedding size : {embeddings.shape[1]}')
print()

print('=== Samples per cancer type ===')
adata = ad.read_h5ad(INPUT_H5AD)
print(f"Reloaded adata: {adata.shape}")
print(f"obs columns: {list(adata.obs.columns)}")
counts = adata.obs['cancer_type'].value_counts()
for cancer, count in counts.items():
    print(f'  {cancer:15s}  {count} patients')
print(f"  {'TOTAL':15s}  {counts.sum()} patients")
print()

print('=== One embedding per cancer type (first 5 values) ===')
for cancer in counts.index:
    mask = adata.obs['cancer_type'] == cancer
    idx = mask.values.argmax()
    preview = embeddings[idx, :5].round(3)
    print(f'  {cancer:15s}  {preview}')

print()
print('No NaN:', not np.isnan(embeddings).any())
print('No Inf:', not np.isinf(embeddings).any())

# Also save cancer predictions if available
if cancer_outputs:
    cancer_emb_path = BASE / 'data/processed/cpgpt_cancer_predictions.npy'
    np.save(cancer_emb_path, cancer_probs)
    print(f'\nCancer predictions saved to: {cancer_emb_path}')

Saved to : /content/drive/MyDrive/multiomics-project/data/processed/cpgpt_embeddings.npy
Shape    : (800, 128)
Any NaN  : False
Any Inf  : False

=== Final Result ===
Total patients : 800
Embedding size : 128

=== Samples per cancer type ===
Reloaded adata: (800, 49156)
obs columns: ['case_id', 'cancer_type', 'file_id']
  TCGA-BRCA        134 patients
  TCGA-LUAD        134 patients
  TCGA-COAD        133 patients
  TCGA-KIRC        133 patients
  TCGA-LIHC        133 patients
  TCGA-THCA        133 patients
  TOTAL            800 patients

=== One embedding per cancer type (first 5 values) ===
  TCGA-BRCA        [ 0.04  -0.041 -0.008 -0.027 -0.115]
  TCGA-LUAD        [ 0.015 -0.042 -0.004 -0.003 -0.08 ]
  TCGA-COAD        [ 0.065 -0.043 -0.021  0.025 -0.114]
  TCGA-KIRC        [-0.081 -0.047 -0.086  0.031 -0.094]
  TCGA-LIHC        [-0.033 -0.052 -0.052  0.046 -0.112]
  TCGA-THCA        [-0.026 -0.045 -0.044 -0.067 -0.104]

No NaN: True
No Inf: True


In [15]:
# Step 9 — Downstream validation: Logistic Regression on CpGPT embeddings
# Same evaluation as the MethylGPT notebook for direct comparison.

import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

base   = f"/content/drive/MyDrive/{SHARED_FOLDER_NAME}/data/processed"
cpgpt  = np.load(f"{base}/cpgpt_embeddings.npy")
labels = np.load(f"{base}/geneformer_labels.npy", allow_pickle=True)

print('CpGPT embeddings:', cpgpt.shape, '| Labels:', labels.shape)

cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
acc = cross_val_score(clf, cpgpt, labels, cv=cv, scoring='accuracy')

print(f'CpGPT-alone accuracy: {acc.mean():.4f} +/- {acc.std():.4f}')
print(f'Random baseline (6 classes): {1/6:.4f}')

CpGPT embeddings: (800, 128) | Labels: (800,)
CpGPT-alone accuracy: 0.8863 +/- 0.0245
Random baseline (6 classes): 0.1667
